### Load datas

In [4]:
import os

docs = []
doc_names = []
for file in os.listdir("asia_data"):
    if file.endswith(".txt"):
        with open(f"asia_data/{file}", "r", encoding="utf-8") as f:
            docs.append(f.read())
            doc_names.append(file)
 
print(f"Loaded {len(docs)} documents for the knowledge base.")

Loaded 9 documents for the knowledge base.


### BM25 for lexical search

In [5]:
from rank_bm25 import BM25Okapi

# BM25 requires each text is tokenized as a (sub)list of words
tokenized_corpus = [doc.lower().split() for doc in docs]
bm25 = BM25Okapi(tokenized_corpus)

def search_bm25(query, top_k=3):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return ranked_indices[:top_k], scores

### Semantic search using Sentence Transformers

In [6]:
from sentence_transformers import SentenceTransformer, util
import torch

model = SentenceTransformer('all-MiniLM-L6-v2')

doc_embed = model.encode(docs,convert_to_tensor=True)

def search_semantic(query, top_k=3):
    query_embed = model.encode(query, convert_to_tensor=True)
    cosine_scores = util.pytorch_cos_sim(query_embed, doc_embed)[0]
    ranked_indices = torch.argsort(cosine_scores, descending=True).tolist()
    return ranked_indices[:top_k], cosine_scores

c:\Homework\Code File\Python Code File\Pratice\Hybrid Semantic-Lexical Search in RAG\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3651.91it/s]


### Implement hybrid search

In [7]:
def hybrid_search(query, top_k=3):
    bm25_ranks, _ = search_bm25(query, top_k=len(docs))
    semantic_ranks, _ = search_semantic(query, top_k=len(docs))

    rrf_scores = {i: 0.0 for i in range(len(docs))}
    k_constant = 60 # standard academic convention

    for rank, doc_i in enumerate(bm25_ranks):
        rrf_scores[doc_i] += 1.0 / (k_constant + rank + 1)


    for rank, doc_i in enumerate(semantic_ranks):
        rrf_scores[doc_i] += 1.0 / (k_constant + rank + 1)

    final_ranked_indices = sorted(rrf_scores.keys(), key=lambda i: rrf_scores[i], reverse=True)

    return final_ranked_indices[:top_k], rrf_scores

### Test

In [10]:
query = "What country has rich culture?"

print(f"--- Query: '{query}' ---")

print("\nTop Semantic Results:")
sem_indices, _ = search_semantic(query, top_k=3)
for i in sem_indices:
    print(f"- {doc_names[i]}")

print("\nTop BM25 Results:")
bm25_indices, _ = search_bm25(query, top_k=3)
for i in bm25_indices:
    print(f"- {doc_names[i]}")

print("\nHybrid Search Results:")
hybrid_indices, _ = hybrid_search(query, top_k=3)
for i in hybrid_indices:
    print(f"- {doc_names[i]}")

--- Query: 'What country has rich culture?' ---

Top Semantic Results:
- Mongolia.txt
- Vietnam.txt
- Thailand.txt

Top BM25 Results:
- Mongolia.txt
- Japan.txt
- Vietnam.txt

Hybrid Search Results:
- Mongolia.txt
- Vietnam.txt
- Japan.txt
